In [3]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [4]:
df = pd.read_parquet("/kaggle/input/datasets/sofiaskz/taxi-2023-2024/my_clean_3_with_weather.parquet")

In [5]:
# datetime
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# время
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

# выходной
df["is_weekend"] = (
    df["pickup_dayofweek"] >= 5
).astype(int)

# часы пик NYC
df["is_rush_hour"] = (
    (
        (df["pickup_hour"] >= 7)
        & (df["pickup_hour"] <= 10)
    )
    |
    (
        (df["pickup_hour"] >= 16)
        & (df["pickup_hour"] <= 19)
    )
).astype(int)

# ночь
df["is_night"] = (
    df["pickup_hour"] <= 5
).astype(int)

# лог расстояния
df["log_distance"] = np.log1p(df["trip_distance"])

# взаимодействия
df["distance_x_hour"] = (
    df["trip_distance"]
    * df["pickup_hour"]
)

# геодистанция
df["geo_distance"] = np.sqrt(
    (df["PU_lat"] - df["DO_lat"])**2
    +
    (df["PU_lon"] - df["DO_lon"])**2
)

In [6]:
df = df[
    (df["duration_min"] >= 2)
    & (df["duration_min"] <= 180)
]

df = df[
    (df["trip_distance"] > 0.1)
    & (df["trip_distance"] < 50)
]

In [7]:
features = [
    "VendorID",
    "passenger_count",

    "trip_distance",

    "PULocationID",
    "DOLocationID",

    "payment_type",

    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "calculated_amount",
    "congestion_surcharge",
    "airport_fee",

    "PU_lon",
    "PU_lat",
    "DO_lon",
    "DO_lat",

    "temperature",
    "precipitation",
    "snowfall",
    "weather_code",

    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",

    "is_weekend",
    "is_rush_hour",
    "is_night",

    "log_distance",
    "distance_x_hour",
    "geo_distance",
]

target = "duration_min"

In [8]:
cat_features = [
    "VendorID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "weather_code",
]

In [9]:
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (6068663, 33)
Test : (1517166, 33)


In [10]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

In [11]:
model5 = CatBoostRegressor(
    iterations=6000,
    learning_rate=0.015,
    depth=8,

    l2_leaf_reg=8,
    random_strength=1.5,
    bagging_temperature=1,

    loss_function="RMSE",
    eval_metric="RMSE",

    early_stopping_rounds=400,

    task_type="GPU",

    verbose=200
)

In [12]:
model5.fit(
    X_train,
    y_train,

    eval_set=(
        X_test,
        y_test
    ),

    cat_features=cat_features,

    use_best_model=True
)

0:	learn: 12.8767992	test: 14.2024140	best: 14.2024140 (0)	total: 7.01s	remaining: 11h 41m 14s
200:	learn: 4.5923278	test: 5.3057253	best: 5.3057253 (200)	total: 1m 17s	remaining: 37m 29s
400:	learn: 4.0167475	test: 4.5811203	best: 4.5811203 (400)	total: 2m 27s	remaining: 34m 17s
600:	learn: 3.7587934	test: 4.2938422	best: 4.2938422 (600)	total: 3m 35s	remaining: 32m 19s
800:	learn: 3.6168652	test: 4.1490929	best: 4.1490929 (800)	total: 4m 44s	remaining: 30m 46s
1000:	learn: 3.5226435	test: 4.0525415	best: 4.0525415 (1000)	total: 5m 53s	remaining: 29m 24s
1200:	learn: 3.4554275	test: 3.9834805	best: 3.9834805 (1200)	total: 7m 2s	remaining: 28m 7s
1400:	learn: 3.4054506	test: 3.9315825	best: 3.9315825 (1400)	total: 8m 10s	remaining: 26m 51s
1600:	learn: 3.3665262	test: 3.8925522	best: 3.8925522 (1600)	total: 9m 19s	remaining: 25m 36s
1800:	learn: 3.3350849	test: 3.8639769	best: 3.8639769 (1800)	total: 10m 27s	remaining: 24m 22s
2000:	learn: 3.3102294	test: 3.8421670	best: 3.8421670 (200

CatBoostRegressor(bagging_temperature=1, depth=8, early_stopping_rounds=400, eval_metric='RMSE', iterations=6000, l2_leaf_reg=8, learning_rate=0.015, loss_function='RMSE', random_strength=1.5, task_type='GPU', verbose=200)

In [13]:
pred5 = model5.predict(X_test)

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

mse5 = mean_squared_error(
    y_test,
    pred5
)

rmse5 = np.sqrt(mse5)

mae5 = mean_absolute_error(
    y_test,
    pred5
)

print("RMSE:", rmse5)
print("MAE :", mae5)

RMSE: 3.699750492909331
MAE : 1.5742423568403399


In [16]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import numpy as np

mse5 = mean_squared_error(
    y_test,
    pred5
)

rmse5 = np.sqrt(mse5)

mae5 = mean_absolute_error(
    y_test,
    pred5
)

r2_5 = r2_score(
    y_test,
    pred5
)

print("RMSE:", rmse5)
print("MAE :", mae5)
print("R2  :", r2_5)

RMSE: 3.699750492909331
MAE : 1.5742423568403399
R2  : 0.9329955111991936


In [17]:
result = pd.DataFrame({
    "actual": y_test.values,
    "predicted": pred5
})

result.head(20)

,actual,predicted
0,9.300000,9.941523
1,22.416667,20.469356
2,6.100000,6.337914
3,15.050000,16.045921
4,40.383333,38.301882
5,6.850000,7.024860
6,21.583333,21.714016
7,29.400000,27.765179
8,18.666667,17.781935
9,7.866667,9.226522


In [18]:
# пример: JFK -> центр Манхэттена

sample_trip = pd.DataFrame([{
    "VendorID": 1,
    "passenger_count": 1,

    "trip_distance": 15.0,

    "PULocationID": 132,   # JFK
    "DOLocationID": 161,   # Midtown Manhattan

    "payment_type": 1,

    "fare_amount": 65.0,
    "extra": 1.0,
    "mta_tax": 0.5,
    "tip_amount": 10.0,
    "tolls_amount": 0.0,
    "improvement_surcharge": 1.0,
    "total_amount": 77.5,
    "calculated_amount": 77.5,
    "congestion_surcharge": 2.5,
    "airport_fee": 1.75,

    "PU_lon": -73.7781,
    "PU_lat": 40.6413,

    "DO_lon": -73.9855,
    "DO_lat": 40.7580,

    "temperature": 20,
    "precipitation": 0,
    "snowfall": 0,
    "weather_code": 0,

    "pickup_hour": 14,
    "pickup_dayofweek": 2,
    "pickup_month": 6,

    "is_weekend": 0,
    "is_rush_hour": 0,
    "is_night": 0,

    "log_distance": np.log1p(15.0),

    "distance_x_hour": 15.0 * 14,

    "geo_distance": np.sqrt(
        (40.6413 - 40.7580)**2
        +
        (-73.7781 - (-73.9855))**2
    )
}])

prediction = model5.predict(sample_trip)

print(
    "Predicted duration:",
    round(prediction[0], 1),
    "minutes"
)

Predicted duration: 78.5 minutes


In [20]:
import joblib

joblib.dump(
    model5,
    "model5.joblib"
)

print("model5 сохранена")

model5 сохранена


In [23]:
import joblib

# сохранить X_test
joblib.dump(
    X_test,
    "X_test.joblib"
)

print("X_test saved")

X_test saved


In [24]:
X_test.to_csv(
    "X_test.csv",
    index=False
)

print("X_test.csv saved")

X_test.csv saved


In [26]:
import pickle

# сохранить модель
with open(
    "model5.pkl",
    "wb"
) as f:
    pickle.dump(
        model5,
        f
    )

print("model5.pkl saved")

model5.pkl saved


In [ ]:
import joblib

# сохранить X_test
joblib.dump(
    y_test,
    "y_test.joblib"
)

print("X_test saved")